In [1]:
#Fresh start for the classifiers

In [2]:
#Import all packages
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.multioutput import MultiOutputClassifier
import warnings
warnings.filterwarnings("ignore", message="unknown class")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# Helper to extract component grams from composite ones

def extract_components(gram_string, paradigms_for_lang):
    """
    Takes a gram string (e.g. 'past participle') and a list of paradigms
    (e.g. [['present','past'], ['participle']]) and returns a list of
    components that occur in the gram.
    """
    gram_string = str(gram_string).lower()
    components = []
    for paradigm in paradigms_for_lang:
        for p in paradigm:
            if p in gram_string:
                components.append(p)
    return sorted(set(components)) if components else ['NONE']

In [4]:
# === Load and prepare data ===

# Load the formatted CSV
df = pd.read_csv(
    "pilot_split.csv",
    dtype=str,
    keep_default_na=False,
    na_values=[]
)

# Define feature and gram columns
FEATURE_COLS = ["TokenID", "Exp", "Res", "Univ", "DTAM", "Evid", "TAML", "NegPol",
                "akt_activity", "akt_accomp", "akt_achiev", "akt_state", "akt_semel"]

GRAM_COLS = ["eng_gram", "deu_gram", "cmn_gram"]

# Quick shape check
print("Data loaded:", df.shape)
print("Columns:", df.columns.tolist()[:10], "...")

Data loaded: (301, 20)
Columns: ['done', 'TokenID', 'target', 'Exp', 'Res', 'Univ', 'DTAM', 'Evid', 'TAML', 'NegPol'] ...


In [5]:
# === Define paradigms and extract components ===

paradigms = {
    'eng': [['ptcp'], ['present', 'past', 'infinitive', 'modal'], ['can', 'could', 'may', 'might', 'must', 'ought_to', 'shall', 'should', 'will', 'would']],
    'deu': [['partizip'], ['inf', 'prasens', 'prat'], ['indikativ', 'k1', 'k2'], ['duerfen', 'koennen', 'moegen', 'muessen', 'sollen', 'wollen'], ['werden'], ['haben', 'sein']],
    'cmn': [['le', 'guo'], ['adverbial'], ['yi', 'yijing', 'zaojiu', 'zaoyi'], ['suo_de'], ['compound'], ['you'], ['nullmarked']],
    'jpn': [['ta', 'te', 'simple'], ['iru'], ['shimau'], ['koto_ga_aru'], ['bakari', 'tokoro'], ['aru']]
}

# === Higher-level paradigm groupings ===
paradigm_groups = {
    'eng': {
        'modal': ['can', 'could', 'may', 'might', 'must', 'ought_to', 'shall', 'should', 'will', 'would']
    },
    'deu': {
        'modal': ['duerfen', 'koennen', 'moegen', 'muessen', 'sollen', 'wollen']
    },
    'cmn': {
        'adverbial': ['yi', 'yijing', 'zaojiu', 'zaoyi']
    }
}

# Make copies to avoid overwriting raw data
df_proc = df.copy()

# Apply component extraction to each gram column
for lg in ['eng', 'deu', 'cmn']:
    col = f"{lg}_gram"
    if col in df_proc.columns:
        df_proc[col] = df_proc[col].apply(lambda x: extract_components(x, paradigms[lg]))

# Quick peek
print(df_proc[[c for c in df_proc.columns if 'gram' in c]].head())

          eng_gram             deu_gram      cmn_gram
0     [past, ptcp]     [partizip, prat]          [le]
1     [past, ptcp]     [partizip, prat]      [suo_de]
2  [present, ptcp]  [partizip, prasens]          [le]
3  [present, ptcp]  [partizip, prasens]      [suo_de]
4     [past, ptcp]     [partizip, prat]  [nullmarked]


In [8]:
# === Evaluation mode per paradigm ===
# keys are tuples that exactly match a paradigm list (order-insensitive; we'll normalize)
EVAL_MODES = {
    'eng': {
        ('ptcp',): 'binary',
        ('present','past','infinitive','modal'): 'multiclass_required',
        ('can','could','may','might','must','ought_to','shall','should','will','would'): 'multiclass_optional',
    },
    'deu': {
        ('partizip',): 'binary',
        ('inf','prasens','prat'): 'multiclass_required',
        ('indikativ', 'kon1', 'kon2'): 'multiclass_required',
        ('modal'): 'binary',
        ('duerfen', 'koennen', 'moegen', 'muessen', 'sollen', 'wollen'): 'multiclass_optional',
        ('werden'): 'binary',
        ('haben', 'sein'): 'multiclass_optional',
    },
    'cmn': {
        ('le','guo'): 'multiclass_optional',
        ('adverbial'): 'binary',
        ('yi','yijing','zaojiu','zaoyi'): 'multiclass_optional',
        ('suo_de'): 'binary',
        ('compound'): 'binary',
        ('you'): 'binary',
        ('nullmarked'): 'binary',
        # Note: 'nullmarked' is a label in the data; we usually *don’t* use it as a paradigm here
    },
    'jpn': {
        ('ta', 'te', 'simple'): 'multiclass_required',
        ('iru'): 'binary',
        ('shimau'): 'binary',
        ('koto_ga_aru'): 'binary',
        ('bakari', 'tokoro'): 'multiclass_optional',
        ('aru'): 'binary'    
}

# helper to normalize a paradigm list into a tuple key
def norm_key(paradigm_list):
    return tuple(sorted(paradigm_list))

In [9]:
# === Helper: generate y_train / y_dev targets according to evaluation mode ===
def build_targets(lang_full, paradigm, mode):
    """
    Returns numeric or categorical targets for this paradigm.

    mode = 'binary'              → 0/1 labels
    mode = 'multiclass_required' → one label from paradigm (string)
    mode = 'multiclass_optional' → one label or '0' for none
    """
    if mode == 'binary':
        # 1 if any element of paradigm occurs, else 0
        y_train = y_train_no_none[lang_full].apply(lambda lst: int(any(p in lst for p in paradigm)))
        y_dev   = y_dev_no_none[lang_full].apply(lambda lst: int(any(p in lst for p in paradigm)))
        return y_train, y_dev

    elif mode == 'multiclass_required':
        # exactly one of the paradigm members should appear
        def get_label(lst):
            for p in paradigm:
                if p in lst:
                    return p
            # if none appear (should be rare)
            return 'MISSING'
        y_train = y_train_no_none[lang_full].apply(get_label)
        y_dev   = y_dev_no_none[lang_full].apply(get_label)
        return y_train, y_dev

    elif mode == 'multiclass_optional':
        # one label if present, otherwise '0' for absence
        def get_label(lst):
            for p in paradigm:
                if p in lst:
                    return p
            return '0'
        y_train = y_train_no_none[lang_full].apply(get_label)
        y_dev   = y_dev_no_none[lang_full].apply(get_label)
        return y_train, y_dev

    else:
        raise ValueError(f"Unknown mode: {mode}")


In [11]:
# === Split into train/dev sets (stratified, reproducible) ===

# Define feature and label subsets from processed dataframe
X = df_proc[["TokenID", "Exp", "Res", "Univ", "DTAM", "Evid", "TAML", "NegPol",
             "akt_activity", "akt_accomp", "akt_achiev", "akt_state", "akt_semel"]]
y = df_proc[["eng_gram", "deu_gram", "cmn_gram"]]

# For stratification, we need a single column of labels per row — use German for now
# (can change this to another language if preferred)
stratify_col = y["deu_gram"].apply(lambda x: "_".join(x))

# --- Handle rare labels (classes with only one example cannot be stratified) ---
vc = stratify_col.value_counts()
rare_labels = vc[vc < 2].index

if len(rare_labels) > 0:
    print(f"⚠️ Removing {len(rare_labels)} rare classes (1 occurrence each).")
    mask = ~stratify_col.isin(rare_labels)
    X = X.loc[mask].reset_index(drop=True)
    y = y.loc[mask].reset_index(drop=True)
    stratify_col = stratify_col.loc[mask].reset_index(drop=True)
    print(f"Remaining samples after cleanup: {len(stratify_col)}")
else:
    print("✅ No rare classes detected — proceeding with full dataset.")

# --- Perform the stratified, reproducible split ---
X_train, X_dev, y_train, y_dev = train_test_split(
    X, y,
    test_size=0.5,
    stratify=stratify_col,
    random_state=42   # ensures reproducible splits
)

print("\nStratified train/dev split complete:")
print("  X_train:", X_train.shape)
print("  X_dev:", X_dev.shape)

⚠️ Removing 1 rare classes (1 occurrence each).
Remaining samples after cleanup: 300

Stratified train/dev split complete:
  X_train: (150, 13)
  X_dev: (150, 13)


In [12]:
# === Filter out rows with 'NONE' or 'n_a' labels (DEU and CMN only) ===

X_train_no_none = {}
X_dev_no_none   = {}
y_train_no_none = {}
y_dev_no_none   = {}

for lang in ["eng_gram", "deu_gram", "cmn_gram"]:
    # Define filtering rule per language
    if lang in ["deu_gram", "cmn_gram"]:
        invalid = lambda lbls: any(lbl in ['NONE', 'n_a'] for lbl in lbls)
    else:
        invalid = lambda lbls: lbls == ['NONE']

    mask_train = ~y_train[lang].apply(invalid)
    mask_dev   = ~y_dev[lang].apply(invalid)
    
    X_train_no_none[lang] = X_train.loc[mask_train].reset_index(drop=True)
    X_dev_no_none[lang]   = X_dev.loc[mask_dev].reset_index(drop=True)
    y_train_no_none[lang] = y_train.loc[mask_train, lang].reset_index(drop=True)
    y_dev_no_none[lang]   = y_dev.loc[mask_dev, lang].reset_index(drop=True)

for lang in y_train_no_none:
    print(f"{lang}: {len(y_train_no_none[lang])} train rows, {len(y_dev_no_none[lang])} dev rows")

eng_gram: 150 train rows, 150 dev rows
deu_gram: 139 train rows, 139 dev rows
cmn_gram: 114 train rows, 105 dev rows


In [ ]:
# === Binarize label sets per language'  ===

"""
General Baseline
These train and evaluate language-level models on the full gram set.
They don’t interfere with the new paradigm logic.
They can be left in as reference / baseline results, or commented out to save runtime.
"""


#mlbs = {}
#y_train_bin = {}
#y_dev_bin   = {}

#for lang in ["eng_gram", "deu_gram", "cmn_gram"]:
#    mlb = MultiLabelBinarizer()
#    y_train_bin[lang] = mlb.fit_transform(y_train_no_none[lang])
#    y_dev_bin[lang]   = mlb.transform(y_dev_no_none[lang])
#    mlbs[lang] = mlb

#    print(f"{lang}: {len(mlb.classes_)} labels → {mlb.classes_}")

In [ ]:
# === Train and evaluate classifiers for each language ===

"""
General Baseline
These train and evaluate language-level models on the full gram set.
They don’t interfere with the new paradigm logic.
They can be left in as reference / baseline results, or commented out to save runtime.
"""

#for lang in ["eng_gram", "deu_gram", "cmn_gram"]:
#    print(f"\n=== Random Forest Classification Report for {lang.upper()} ===")

    # Initialize a classifier (can swap this for MLPClassifier later)
#    clf = RandomForestClassifier(random_state=42)

    # Train on non-NONE data for this language
#    X_train_lang = X_train_no_none[lang].drop(columns=["TokenID"])
#    X_dev_lang   = X_dev_no_none[lang].drop(columns=["TokenID"])

#   clf.fit(X_train_lang, y_train_bin[lang])
#    preds = clf.predict(X_dev_lang)

    # Decode predictions back to label lists
#    pred_sets = mlbs[lang].inverse_transform(preds)
#    true_sets = y_dev_no_none[lang]

    # Build comparable string lists for easy accuracy checks
#    true_vals = pd.Series([", ".join(t) for t in true_sets])
#    pred_vals = pd.Series([", ".join(p) for p in pred_sets])

    # Manual accuracy
#    accuracy = (true_vals == pred_vals).mean()
#    print(f"Manual accuracy: {accuracy:.3f}")

    # Detailed report
#    print(classification_report(true_vals, pred_vals, zero_division=0))

In [13]:
# === Build paradigm-specific subsets (testing stage) ===

for lang_short, lang_full in zip(['eng', 'deu', 'cmn'], ['eng_gram', 'deu_gram', 'cmn_gram']):
    print(f"\n=== Paradigm subsets for {lang_short.upper()} ===")
    for paradigm in paradigms[lang_short]:
        pname = "_".join(paradigm)  # just for readable naming
        mask = y_dev_no_none[lang_full].apply(lambda lst: any(p in lst for p in paradigm))
        subset_size = mask.sum()
        print(f"{pname:<20} → {subset_size} tokens in dev set")


=== Paradigm subsets for ENG ===
ptcp                 → 150 tokens in dev set
present_past_infinitive_modal → 143 tokens in dev set
can_could_may_might_must_ought_to_shall_should_will_would → 7 tokens in dev set

=== Paradigm subsets for DEU ===
partizip             → 123 tokens in dev set
inf_prasens_prat     → 139 tokens in dev set
indikativ_k1_k2      → 17 tokens in dev set
duerfen_koennen_moegen_muessen_sollen_wollen → 0 tokens in dev set
werden               → 0 tokens in dev set
haben_sein           → 0 tokens in dev set

=== Paradigm subsets for CMN ===
le_guo_you_nullmarked → 83 tokens in dev set
yi_yijing_zaojiu_zaoyi → 13 tokens in dev set
suo_de               → 14 tokens in dev set
compound             → 0 tokens in dev set


In [14]:
# === Train and evaluate within each paradigm subset (mode-aware) ===

for lang_short, lang_full in zip(['eng', 'deu', 'cmn'], ['eng_gram', 'deu_gram', 'cmn_gram']):
    print(f"\n=== Paradigm-level classification for {lang_short.upper()} ===")
    for paradigm in paradigms[lang_short]:
        pname = "_".join(paradigm)
        mode = EVAL_MODES[lang_short].get(norm_key(paradigm), 'multiclass_optional')

        # === Build y and optionally subset X according to evaluation mode ===
        if mode == 'binary':
            # Keep ALL rows, label 1 if paradigm element present
            y_train_sub, y_dev_sub = build_targets(lang_full, paradigm, mode)
            X_train_sub = X_train_no_none[lang_full].drop(columns=["TokenID"])
            X_dev_sub   = X_dev_no_none[lang_full].drop(columns=["TokenID"])

        else:
            # Subset to rows containing at least one paradigm element
            mask_train = y_train_no_none[lang_full].apply(lambda lst: any(p in lst for p in paradigm))
            mask_dev   = y_dev_no_none[lang_full].apply(lambda lst: any(p in lst for p in paradigm))

            X_train_sub = X_train_no_none[lang_full].loc[mask_train].drop(columns=["TokenID"])
            X_dev_sub   = X_dev_no_none[lang_full].loc[mask_dev].drop(columns=["TokenID"])

            # Build y using same subset
            y_train_sub, y_dev_sub = build_targets(lang_full, paradigm, mode)
            y_train_sub = y_train_sub.loc[mask_train].reset_index(drop=True)
            y_dev_sub   = y_dev_sub.loc[mask_dev].reset_index(drop=True)

        # Skip tiny sets (fewer than 5 instances)
        if len(y_train_sub) < 5 or len(y_dev_sub) < 5:
            continue

        # === Train classifier ===
        clf = RandomForestClassifier(random_state=42)
        clf.fit(X_train_sub, y_train_sub)
        preds = clf.predict(X_dev_sub)

        # === Evaluate ===
        print(f"\n  {pname:<35} → {len(y_dev_sub):>3} dev tokens | mode = {mode}")

        if mode == 'binary':
            acc = (preds == y_dev_sub).mean()
            print(f"    Binary accuracy = {acc:.3f}")
        else:
            print(classification_report(y_dev_sub, preds, zero_division=0))


=== Paradigm-level classification for ENG ===

  ptcp                                → 150 dev tokens | mode = binary
    Binary accuracy = 1.000

  present_past_infinitive_modal       → 143 dev tokens | mode = multiclass_optional
              precision    recall  f1-score   support

        past       0.97      0.90      0.93        39
     present       0.96      0.99      0.98       104

    accuracy                           0.97       143
   macro avg       0.97      0.94      0.95       143
weighted avg       0.97      0.97      0.96       143


  can_could_may_might_must_ought_to_shall_should_will_would →   7 dev tokens | mode = multiclass_optional
              precision    recall  f1-score   support

       could       0.50      1.00      0.67         1
      should       0.00      0.00      0.00         2
       would       0.80      1.00      0.89         4

    accuracy                           0.71         7
   macro avg       0.43      0.67      0.52         7
weighted

In [ ]:
# === Confusion Matrices per language (Random Forest baseline) ===

for lang in ["eng_gram", "deu_gram", "cmn_gram"]:
    print(f"\n=== Confusion Matrix for {lang.upper()} ===")

    # True and predicted values
    true_sets = y_dev_no_none[lang]
    pred_sets = mlbs[lang].inverse_transform(
        RandomForestClassifier(random_state=42)
        .fit(
            X_train_no_none[lang].drop(columns=["TokenID"]),
            y_train_bin[lang]
        )
        .predict(X_dev_no_none[lang].drop(columns=["TokenID"]))
    )

    # Flatten labels to strings for readability
    true_vals = [", ".join(t) for t in true_sets]
    pred_vals = [", ".join(p) for p in pred_sets]

    # Build confusion matrix
    labels = sorted(set(true_vals + pred_vals))
    cm = confusion_matrix(true_vals, pred_vals, labels=labels)

    cm_df = pd.DataFrame(cm, index=[f"True_{l}" for l in labels],
                              columns=[f"Pred_{l}" for l in labels])
    print(cm_df)


In [ ]:
# === Export confusion matrices to CSV ===
OUTPUT_DIR = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Pilot Resplit/Outputs"

for lang in ["eng_gram", "deu_gram", "cmn_gram"]:
    # True and predicted values
    true_sets = y_dev_no_none[lang]
    pred_sets = mlbs[lang].inverse_transform(
        RandomForestClassifier(random_state=42)
        .fit(
            X_train_no_none[lang].drop(columns=["TokenID"]),
            y_train_bin[lang]
        )
        .predict(X_dev_no_none[lang].drop(columns=["TokenID"]))
    )

    true_vals = [", ".join(t) for t in true_sets]
    pred_vals = [", ".join(p) for p in pred_sets]

    labels = sorted(set(true_vals + pred_vals))
    cm = confusion_matrix(true_vals, pred_vals, labels=labels)

    cm_df = pd.DataFrame(cm, index=[f"True_{l}" for l in labels],
                              columns=[f"Pred_{l}" for l in labels])
    cm_path = f"{OUTPUT_DIR}/{lang}_confusion_matrix.csv"
    cm_df.to_csv(cm_path)
    print("Exported:", cm_path)